In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Data Ingestion
# Note: Download 'heart.csv' from the UCI Machine Learning Repository and place it in the 'data/' folder.
print("Loading dataset...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]

df = pd.read_csv(url, names=columns)

# Clean missing values (UCI dataset uses '?' for missing values in 'ca' and 'thal')
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)

# Convert columns to numeric
df = df.apply(pd.to_numeric)

# The target variable contains values from 0 to 4. 
# We will convert this to a binary classification problem: 0 = No Disease, 1 = Disease present (1, 2, 3, 4)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

# 2. Exploratory Data Analysis (EDA)
print("Generating EDA Visualizations...")
plt.figure(figsize=(10, 6))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Heart Disease Presence (0 = No, 1 = Yes)')
plt.savefig('heart_disease_distribution.png')
plt.close()

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.savefig('correlation_matrix.png')
plt.close()

# 3. Data Preprocessing
print("Preprocessing data...")
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Model Building
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# 5. Model Evaluation
print("Evaluating Model...")
y_pred = rf_model.predict(X_test_scaled)

print("\n--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix Visualization
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('confusion_matrix.png')
plt.close()

# Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Feature Importances in Predicting Heart Disease")
plt.bar(range(X.shape[1]), importances[indices], align="center", color='teal')
plt.xticks(range(X.shape[1]), [columns[i] for i in indices], rotation=45)
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

print("Pipeline execution complete! Visualizations saved to your directory.")


Loading dataset...
Generating EDA Visualizations...


C:\Users\NEELAM\AppData\Local\Temp\ipykernel_25852\796953855.py:34: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='target', data=df, palette='Set2')


Preprocessing data...
Training Random Forest Classifier...
Evaluating Model...

--- Model Performance ---
Accuracy: 0.87

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.91      0.88        32
           1       0.88      0.82      0.85        28

    accuracy                           0.87        60
   macro avg       0.87      0.86      0.87        60
weighted avg       0.87      0.87      0.87        60

Pipeline execution complete! Visualizations saved to your directory.


In [2]:
import pickle

# Define the filename
model_filename = 'heart_disease_rf_model.pkl'

# Save both the trained model and the scaler as a dictionary
with open(model_filename, 'wb') as file:
    pickle.dump({'model': rf_model, 'scaler': scaler}, file)
    
print(f"✅ Model and scaler successfully saved as {model_filename}!")


✅ Model and scaler successfully saved as heart_disease_rf_model.pkl!
